In [28]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from functools import lru_cache
from collections import defaultdict


## __Open Visum .VER__

In [29]:
#Open network .ver file (from local disk not onedrive)
import win32com.client

#Visum = com.Dispatch("Visum.Visum.250") #Add .250 for Visum 25 version
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Visum Projects\RedOSMNX_AMG_24 - Copy.ver')
C = win32com.client.constants

In [39]:
Net = Visum.Net

## __Read Stop Points IMEPLAN__ (snapped)

In [30]:
# Need correction (must be 12,518) & agreed ox SP work better
bus_stops_buffer16 = gpd.read_file(r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Insumos OSMNX\Transporte Publico\Snapped\buffer_and_hierarchy\bus_stops_snapped_buffer16.shp')
bus_stops_ox_noservice = gpd.read_file(r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Insumos OSMNX\Transporte Publico\Snapped\ox_nearestEdge\bus_stops_snapped_ox.shp')
bus_stops_ox_position_processed = gpd.read_file(r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Insumos OSMNX\Transporte Publico\Snapped\ox_nearestEdge\RelPos preprocess\bus_stops_ox_relpos_preprocessed.shp')
bus_stops_oxVisum_position_processed = gpd.read_file(r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Visum Objects\Bus Stops Snapped to Visum Link\bus_stops_snapped.shp')

visum_links = gpd.read_file(r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Insumos OSMNX\Visum\Edges\edges_visum.shp')

In [31]:
bus_stops_oxVisum_position_processed['NearestLin'].value_counts()

NearestLin
['residential']                     5104
['tertiary']                        3731
['secondary']                       1711
['primary']                         1312
['trunk']                            230
['unclassified']                     225
['service']                           44
['trunk_link']                        43
['primary_link']                      41
['living_street']                     18
['motorway_link']                     16
['tertiary_link']                     13
['secondary_link']                    12
['motorway']                          11
['tertiary', 'secondary']              2
['secondary_link', 'secondary']        2
['motorway_link', 'residential']       1
['trunk_link', 'residential']          1
['tertiary', 'trunk_link']             1
Name: count, dtype: int64

In [32]:
bus_stops = bus_stops_oxVisum_position_processed.copy()

### Transform bus stops CRS to EPSG:4326 to insert physical stop on x,y (Visum is in 4326)

In [33]:
print(f"Original bus stops CRS: {bus_stops.crs}")
bus_stops.to_crs(crs="EPSG:4326", inplace=True)
print(f"Updated bus stops CRS: {bus_stops.crs}")
bus_stops

Original bus stops CRS: EPSG:32613
Updated bus stops CRS: EPSG:4326


,id,stop_id,XCoord,YCoord,LinkNo,SnapDist_m,FromNodeNo,ToNodeNo,RelPosOnLi,NearestLin,route_list,SP_id,geometry
0,1.0,1,680976.456130,2.272737e+06,9238.0,7.332037,3304.0,30379.0,0.108069,['secondary'],"C116-V1,C116-V2,C15,N/A,T13A-C02,T13A-C03,Tron...",473,POINT (-103.26406 20.54483)
1,2.0,2,681483.253691,2.280846e+06,4352.0,6.928147,2340.0,1495.0,0.159889,['primary'],"C106,C19,C46-V1,C46-V2,N/A,Troncal 19 Periferico",267,POINT (-103.25837 20.61802)
2,3.0,3,700561.553323,2.281164e+06,65242.0,3.147165,24808.0,24811.0,0.824667,['residential'],"N/A,R1-Tepetates,R2-Zorrillos,T21 Zapotlanejo,...",1331,POINT (-103.07532 20.61896)
3,4.0,4,700707.084499,2.281320e+06,8267.0,10.889076,2947.0,30220.0,0.410244,['secondary'],"N/A,R1-Tepetates,R2-Zorrillos,T21 Zapotlanejo,...",432,POINT (-103.07391 20.62035)
4,5.0,5,700907.289266,2.281378e+06,8384.0,7.604439,2997.0,2948.0,0.605914,['secondary'],"N/A,R1-Tepetates,R2-Zorrillos,T21 Zapotlanejo,...",445,POINT (-103.07198 20.62085)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12513,NaN,12514,699396.250113,2.280168e+06,49413.0,10.040549,18239.0,18245.0,0.108508,['primary'],None,1157,POINT (-103.08661 20.61009)
12514,NaN,12515,699836.994172,2.280555e+06,49451.0,0.302038,18255.0,25555.0,0.900000,['primary'],None,1159,POINT (-103.08234 20.61354)
12515,NaN,12516,701461.660841,2.281479e+06,8288.0,2.373914,2957.0,2959.0,0.791024,['secondary'],None,434,POINT (-103.06665 20.6217)
12516,NaN,12517,701108.581247,2.281835e+06,8511.0,7.931644,3047.0,3072.0,0.824228,['tertiary'],None,449,POINT (-103.07 20.62496)


## __Insert bus stops in Visum__

In [34]:
# Links
links = Visum.Net.Links
link_ids = links.GetMultiAttValues("No")  # ID del enlace
from_node = links.GetMultiAttValues("FromNodeNo")
to_node = links.GetMultiAttValues("ToNodeNo")
lenght_km = links.GetMultiAttValues("Length")
tsysset = links.GetMultiAttValues("TSysSet")

visum_links = pd.DataFrame({
    "LinkNo": [l[1] for l in link_ids],
    "FromNode": [fn[1] for fn in from_node],
    "ToNode": [tn[1] for tn in to_node],
    "Length": [l[1] for l in lenght_km],
    "TSysSet": [tsys[1] for tsys in tsysset]
})

visum_links

,LinkNo,FromNode,ToNode,Length,TSysSet
0,1.0,1.0,103492.0,4.827825,"B,C"
1,1.0,103492.0,1.0,0.000000,
2,2.0,1.0,15708.0,0.146029,"B,C"
3,2.0,15708.0,1.0,0.000000,
4,3.0,2.0,3.0,0.520237,"B,C"
...,...,...,...,...,...
593963,500924.0,213454.0,205191.0,0.010734,"B,C,W"
593964,500925.0,206297.0,213455.0,0.000000,
593965,500925.0,213455.0,206297.0,0.010452,TL
593966,500926.0,206297.0,213456.0,0.000000,


In [35]:
visum_links[visum_links['TSysSet'] == '']

,LinkNo,FromNode,ToNode,Length,TSysSet
1,1.0,103492.0,1.0,0.0,
3,2.0,15708.0,1.0,0.0,
5,3.0,3.0,2.0,0.0,
7,4.0,3068.0,2.0,0.0,
9,5.0,7341.0,3.0,0.0,
...,...,...,...,...,...
593958,500922.0,205191.0,213452.0,0.0,
593961,500923.0,213454.0,213453.0,0.0,
593962,500924.0,205191.0,213454.0,0.0,
593964,500925.0,206297.0,213455.0,0.0,


In [11]:
x = visum_links.loc[(visum_links["FromNode"] == 1) & (visum_links["ToNode"] == 103492)]
x['Length']

0    4.827825
Name: Length, dtype: float64

In [36]:
# Function to check if a stop with the same ID already exists in Visum
def visum_SP_exists(stop_id):
    try:
        return Net.StopPoints.ItemByKey(stop_id) is not None
    except Exception:
        return False

def opposite_dir(fromNode, toNode):
    oppositeLink = visum_links.loc[(visum_links["FromNode"] == toNode) & (visum_links["ToNode"] == fromNode)] #get the opp. dir link of the one that holds the snap
    valid = oppositeLink.loc[
        (oppositeLink['Length'] > 0) &
        (oppositeLink['TSysSet'] != '')
    ]

    return not valid.empty

In [37]:
print(opposite_dir(1, 103492))

False


In [27]:
bus_stops.columns

Index(['id', 'stop_id', 'XCoord', 'YCoord', 'LinkNo', 'SnapDist_m',
       'FromNodeNo', 'ToNodeNo', 'RelPosOnLi', 'NearestLin', 'route_list',
       'SP_id', 'geometry'],
      dtype='object')

In [41]:
stops_not_inserted = []
inserted_sp = set()
sp_counter = 1

for SP in bus_stops.itertuples():
    # 1) Extract SP attributes
    stop_id = SP.stop_id #1 - 12,518
    id_imeplan = SP.id if not pd.isna(SP.id) else 0

    sp_id = SP.SP_id #1 - 11,400 (unificado por link)
    nearest_link = SP.LinkNo
    from_node = SP.FromNodeNo
    to_node = SP.ToNodeNo
    relative_position = SP.RelPosOnLi
    routes = SP.route_list
    #relpos_adjusted = SP.RelPos_New
    x = SP.geometry.x
    y = SP.geometry.y

    # 2) Add Stop & StopArea to Visum (12,518 physical stops)
    try:
        stop = Net.AddStop(stop_id, x, y)
        stop_area = Net.AddStopArea(stop_id, stop_id, from_node, x, y)

    except Exception as e:
        print(f"Error inserting {stop_id}: {e}")
        stops_not_inserted.append(stop_id)
        continue

    if sp_id not in inserted_sp:
        try:
            stop_point = Net.AddStopPointOnLink(sp_counter, stop_id, from_node, to_node, True)
            sp_counter += 1
            try:
                stop_point.SetAttValue("RelPos", relative_position)
                stop_point.SetAttValue('STOP_ID_IMEPLAN', id_imeplan)
                stop_point.SetAttValue('ROUTES', routes)
                stop_point.SetAttValue("TSysSet", "B")
            except Exception as e:
                print(f'Error setting attributes for {sp_id}: {e}')

            if opposite_dir(from_node, to_node):
                stop_point2 = Net.AddStopPointOnLink(sp_counter, stop_id, to_node, from_node, True)
                sp_counter += 1

                try:
                    stop_point2.SetAttValue("RelPos", 1-relative_position) #for it to be in the same position of opp. link
                    stop_point2.SetAttValue('STOP_ID_IMEPLAN', id_imeplan)
                    stop_point2.SetAttValue('ROUTES', routes)
                    stop_point2.SetAttValue("TSysSet", "B")
                except Exception as e:
                    print(f'Error setting attributes for {sp_id} partner: {e}')

            inserted_sp.add(sp_id)

        except Exception as e:
            print(f"Error inserting stop point for {stop_id}: {e}")
            stops_not_inserted.append(stop_id)
            continue
   
    stop.SetAttValue('STOP_ID_IMEPLAN', id_imeplan)
    

### __Update bus stops attributes in Visum__

In [149]:
route_numeric_attrs = {
    'mobiliario':'mobiliario',
    'iluminaci':'iluminacion',
    'señal_ver':'senal_ver',
    'señal_hor':'senal_hor',
    'banqueta':'banqueta',
    'vegetació':'vegetacion'
}

stop_points = Visum.Net.StopPoints


for SP in bus_stops.itertuples():
    # 1) Get SP from visum using stop_id
    stop_id = SP.stop_id
    stop_point = stop_points.ItemByKey(stop_id)

    # 2) Set routes attribute in stop_points
    for i in range(1,36):
        route=getattr(SP, f"ruta_{i}", None)
        if pd.notnull(route) and route != '' and route != 'N/A':
            stop_point.SetAttValue(f"ruta_{i}", route)

    # 3) Set numeric attributes in stop_points
    for col, visum_attr in route_numeric_attrs.items():
        value = getattr(SP, col, None)
        if pd.isna(value):
            value = 0
        stop_point.SetAttValue(visum_attr, value)

### Open them only for BUS (B) all so far which are the 12,518 are bus stops

In [150]:
for sp in Visum.Net.StopPoints:
    sp.SetAttValue("TSysSet", "B")

### Relative positions of existing SP per link
- to see if a new SP to be inserted conflicts with pre-existing SP in the link

In [79]:
# SP existentes por link
link_sp_positions = defaultdict(list)

for sp in Net.StopPoints:
    link_id = sp.AttValue("LinkNo")
    rel_pos = sp.AttValue("RelPos")

    link_sp_positions[link_id].append(rel_pos)

link_sp_positions

defaultdict(list, {})

### Neighbor links per Node
- to see if new SP to be inserted is too close to a SP of the neighboring link

In [27]:
node_in_links = defaultdict(list)
node_out_links = defaultdict(list)

for link in Net.Links:
    link_id = link.AttValue("No")
    from_n = link.AttValue("FromNodeNo")
    to_n = link.AttValue("ToNodeNo")

    key = (link_id, from_n, to_n)

    node_out_links[from_n].append(key)
    node_in_links[to_n].append(key)

In [28]:
node_in_links

defaultdict(list,
            {103492.0: [(1.0, 1.0, 103492.0),
              (199578.0, 78634.0, 103492.0),
              (258630.0, 47861.0, 103492.0)],
             1.0: [(1.0, 103492.0, 1.0),
              (2.0, 15708.0, 1.0),
              (59898.0, 22510.0, 1.0)],
             15708.0: [(2.0, 1.0, 15708.0),
              (42957.0, 21829.0, 15708.0),
              (58143.0, 21824.0, 15708.0)],
             3.0: [(3.0, 2.0, 3.0),
              (5.0, 7341.0, 3.0),
              (68762.0, 26341.0, 3.0)],
             2.0: [(3.0, 3.0, 2.0),
              (4.0, 3068.0, 2.0),
              (79712.0, 31294.0, 2.0)],
             3068.0: [(4.0, 2.0, 3068.0),
              (8574.0, 26414.0, 3068.0),
              (8576.0, 3069.0, 3068.0)],
             7341.0: [(5.0, 3.0, 7341.0),
              (20318.0, 7342.0, 7341.0),
              (20319.0, 7340.0, 7341.0)],
             9626.0: [(6.0, 4.0, 9626.0),
              (26657.0, 9627.0, 9626.0),
              (26658.0, 25881.0, 9626.0),
    

In [58]:
# Function to check if a new stop point can be added to a link without conflicting with existing stop points
def is_valid_position(link_id, rel_pos, threshold=0.001):
    existing = link_sp_positions.get(link_id, [])
    return not any(abs(rel_pos - p) < threshold for p in existing)

# Function to check if a stop with the same ID already exists in Visum
def visum_stop_exists(stop_id):
    try:
        return Net.Stops.ItemByKey(stop_id) is not None
    except Exception:
        return False

In [59]:
def conflicts_with_neighbors(link_id, rel_pos, threshold=0.001):
    selected_edge = visum_links[visum_links['LinkNo'] == link_id]
    # inicio del link → revisar link anterior
    if rel_pos < threshold:
        for sp in Net.StopPoints:
            if sp.AttValue("ToNodeNo") == selected_edge.FromNodeNo:
                if abs(1 - sp.AttValue("RelPos")) < threshold:
                    return True

    # final del link → revisar siguiente link
    if rel_pos > 1 - threshold:
        for sp in Net.StopPoints:
            if sp.AttValue("FromNodeNo") == selected_edge.ToNodeNo:
                if abs(sp.AttValue("RelPos")) < threshold:
                    return True

    return False

## __Create StopPoints on Visum__

In [67]:
bus_stops

,id,mobiliario,iluminaci,señal_ver,señal_hor,banqueta,vegetació,ruta_1,ruta_2,ruta_3,...,NearestNod,NearestN_1,NearestEdg,SnapDist_m,NearestLin,RelPosOnLi,FromNodeNo,ToNodeNo,NearestL_1,geometry
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Troncal 13A Solidaridad,Troncal 13B Solidaridad,T13A-C02,...,1771858665,3304,"(1771858665, 8778745267, 0)",8.207127,9238,0.116142,3304,30379,secondary,POINT (-103.26406 20.54483)
1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,Troncal 19 Periferico,C19,C46-V1,...,1700484749,2340,"(1677124239, 1700484749, 0)",8.441228,4352,0.817699,1495,2340,primary,POINT (-103.25837 20.61802)
2,3.0,0.0,1.0,0.0,0.0,1.0,0.0,T21 Zapotlanejo,T21-C01,R1-Tepetates,...,8407044728,24811,"(8407044724, 8407044728, 0)",6.982640,65242,0.850051,24808,24811,residential,POINT (-103.07532 20.61896)
3,4.0,0.0,1.0,0.0,0.0,1.0,0.0,T21 Zapotlanejo,T21-C01,R1-Tepetates,...,1746083332,2947,"(1746083332, 8778413347, 0)",10.527186,8267,0.339513,2947,30220,secondary,POINT (-103.07391 20.62035)
4,5.0,0.0,1.0,0.0,0.0,1.0,0.0,T21 Zapotlanejo,T21-C01,R1-Tepetates,...,1746083335,2948,"(1746090115, 1746083335, 0)",4.600218,8384,0.595657,2997,2948,secondary,POINT (-103.07198 20.62085)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12513,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,...,5987144952,16245,"(6322033098, 6322033110, 0)",8.663037,49413,0.110336,18239,18245,primary,POINT (-103.08661 20.61009)
12514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,...,8414023283,25555,"(6322039422, 8414023283, 0)",1.691836,49451,0.920860,18255,25555,primary,POINT (-103.08234 20.61354)
12515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,...,1746083377,2959,"(1746083353, 1746083377, 0)",2.215223,8288,0.801377,2957,2959,secondary,POINT (-103.06665 20.6217)
12516,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,...,1746293814,3072,"(1746195132, 1746293814, 0)",5.281715,8511,0.772637,3047,3072,tertiary,POINT (-103.07 20.62496)


In [82]:
THRESH = 0.002
EDGE_EPS = 0.01   # para evitar bordes
NUDGE = 0.002     # para empujar posiciones

In [61]:
def clamp_position(rel):
    if rel < EDGE_EPS:
        return EDGE_EPS
    if rel > 1 - EDGE_EPS:
        return 1 - EDGE_EPS
    return rel

In [62]:
def conflict_same_link(link_id, rel):
    # Get relative positions of existing stop points on the same link
    # Check if the new stop point's relative position is too close to any existing stop point on the same link
    return any(abs(rel - p) < THRESH for p in link_sp_positions.get(link_id, []))

In [63]:
def conflict_neighbors(from_node, to_node, rel):
    #link = Net.Links.ItemByKey(link_id)
   # from_n = link.AttValue("FromNodeNo")
    #to_n = link.AttValue("ToNodeNo")

    # inicio
    if rel < THRESH:
        for l in node_in_links[from_node]:
            for p in link_sp_positions.get(l, []):
                if abs(1 - p) < THRESH:
                    return True

    # final
    if rel > 1 - THRESH:
        for l in node_out_links[to_node]:
            for p in link_sp_positions.get(l, []):
                if abs(p) < THRESH:
                    return True

    return False

In [64]:
def conflicts_with_neighbors(from_node, to_node, rel_pos, threshold=0.001):
    # inicio del link → revisar link anterior
    if rel_pos < threshold:
        for sp in Net.StopPoints:
            if sp.AttValue("ToNodeNo") == from_node:
                if abs(1 - sp.AttValue("RelPos")) < threshold:
                    return True

    # final del link → revisar siguiente link
    if rel_pos > 1 - threshold:
        for sp in Net.StopPoints:
            if sp.AttValue("FromNodeNo") == to_node:
                if abs(sp.AttValue("RelPos")) < threshold:
                    return True

    return False

In [78]:
def find_valid_position(stop_id, link_id, from_node, to_node, rel):
    rel = clamp_position(rel)
    position_adjusted = False

    # si la posición inicial está cerca de 0.5, sal de esa zona
    if abs(rel - 0.5) <= THRESH:
        rel = 0.5 + NUDGE if rel >= 0.5 else 0.5 - NUDGE
        rel = clamp_position(rel)

    for _ in range(10):  # máximo 10 intentos
        if not conflict_same_link(link_id, rel) and not conflict_neighbors(from_node, to_node, rel) and abs(rel-0.5) > THRESH:
            if position_adjusted:
                print(f'Relative position had to be adjusted for SP:{stop_id}')
            return rel
        
        rel += NUDGE
        rel = clamp_position(rel)
        position_adjusted = True

       # if rel > 1 - EDGE_EPS:
        #    rel = 0.5  # fallback al centro
        #    break

    print(f'Couldnt adjust position for SP: {stop_id}')
    return None  # no se pudo arreglar

In [76]:
bus_stops = bus_stops.sort_values(["NearestLin", "RelPosOnLi"])

In [85]:
THRESH = 0.001

def adjust_positions(group):
    positions = group["RelPosOnLi"].values.copy()
    
    for i in range(1, len(positions)):
        if positions[i] - positions[i-1] < THRESH:
            positions[i] = positions[i-1] + THRESH
    
    # clamp final
    positions = positions.clip(0.01, 0.99)
    
    group["RelPosAdj"] = positions
    return group

bus_stops_forVisum = bus_stops.groupby("NearestLin", group_keys=False).apply(adjust_positions)

C:\Users\AP03542515\AppData\Local\Temp\3\ipykernel_68624\3419638051.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bus_stops_forVisum = bus_stops.groupby("NearestLin", group_keys=False).apply(adjust_positions)


In [86]:
bus_stops_forVisum

,id,mobiliario,iluminaci,señal_ver,señal_hor,banqueta,vegetació,ruta_1,ruta_2,ruta_3,...,NearestN_1,NearestEdg,SnapDist_m,NearestLin,RelPosOnLi,FromNodeNo,ToNodeNo,NearestL_1,geometry,RelPosAdj
5854,5946.0,0.0,0.0,0.0,0.0,0.0,0.0,T13A-C01,ATQ-IXT,Caj-SJuan,...,9,"(325713635, 6584813561, 0)",10.119210,17,0.048216,9,19477,primary,POINT (-103.25134 20.45556),0.048216
5852,5944.0,0.0,0.0,0.0,0.0,0.0,0.0,T13A-C01,Caj-SJuan,N/A,...,19477,"(325713635, 6584813561, 0)",6.213091,17,0.891631,9,19477,primary,POINT (-103.2519 20.45481),0.891631
4482,4523.0,0.0,0.0,0.0,0.0,1.0,0.0,Troncal 13C Solidaridad,T13C-C01,T13C-C02,...,25112,"(325722532, 8408128650, 0)",7.657755,29,0.711381,14,25112,secondary,POINT (-103.18825 20.57282),0.711381
7672,7817.0,1.0,0.0,0.0,0.0,0.0,0.0,Troncal 13C Solidaridad,T13C-C02,T13C-C03,...,15,"(325722722, 8784590905, 0)",8.944722,33,0.130065,15,30539,secondary,POINT (-103.18431 20.55397),0.130065
7783,7929.0,1.0,0.0,0.0,0.0,0.0,0.0,Troncal 13C Solidaridad,T13C-C02,T13C-C03,...,21303,"(325723400, 6821046163, 0)",6.811722,43,0.728447,19,21303,secondary,POINT (-103.1821 20.54415),0.728447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12034,12230.0,0.0,1.0,0.0,1.0,1.0,0.0,Troncal 11A Rio Nilo Guadalupe,T11A-C01,T11A-C02,...,134514,"(13542846568, 1343692894, 0)",10.244669,491835,0.663771,204958,134514,residential,POINT (-103.37655 20.67074),0.663771
10310,10480.0,0.0,0.0,0.0,0.0,0.0,0.0,Troncal 11A Rio Nilo Guadalupe,T11A-C01,T11A-C02,...,204967,"(13542914042, 1343692881, 0)",8.630558,491863,0.462211,204967,134511,residential,POINT (-103.37553 20.67227),0.462211
3558,3588.0,1.0,1.0,1.0,1.0,1.0,1.0,Troncal 12 Vallarta,N/A,N/A,...,204975,"(13542914075, 8311171474, 0)",9.529101,491886,0.332679,204975,184313,secondary,POINT (-103.37771 20.67451),0.332679
10309,10479.0,0.0,0.0,0.0,0.0,0.0,0.0,Troncal 11A Rio Nilo Guadalupe,T11A-C01,T11A-C02,...,176263,"(13542921732, 1342831090, 0)",9.094165,491945,0.853837,204994,134466,residential,POINT (-103.37981 20.67225),0.853837


In [83]:

stops_not_inserted = []
stops_id_null = []

for stop in bus_stops.itertuples():
    stop_id = stop.id
    nearest_node = stop.NearestN_1
    nearest_link = stop.NearestLin
    from_node = stop.FromNodeNo
    to_node = stop.ToNodeNo
    relative_position = stop.RelPosOnLi
    x = stop.geometry.x
    y = stop.geometry.y

    # 1) Check if stop already exists
    if visum_stop_exists(stop_id):
        #print(f"STOP {stop_id} already exists in Visum, skipping.")
        continue

    # 2) If SP id is null
    if pd.isna(stop_id):
        stops_id_null.append(nearest_link)
        continue

    # 3) Use relative position OR adjust it if its too close to other SP
    rel_valid = find_valid_position(stop_id, nearest_link, from_node, to_node, relative_position)

    if rel_valid is None:
        print(f"Skipping {stop_id} - no valid position found")
        stops_not_inserted.append(stop_id)
        continue


    # 4) Add SP to Visum
    try:
        stop = Net.AddStop(stop_id, x, y)
        stop_area = Net.AddStopArea(stop_id, stop_id, nearest_node, x, y)
        stop_point = Net.AddStopPointOnLink(stop_id, stop_id, from_node, to_node, True)
    except Exception as e:
        print(f"Error inserting {stop_id}: {e}")
        stops_not_inserted.append(stop_id)
        continue

    # Change relative position of SP just created
    try:
        stop_point.SetAttValue("RelPos", rel_valid)
        # 🔁 actualizar memoria
        link_sp_positions[link_id].append(rel_valid)
    except Exception as e:
        print(f"RelPos failed for {stop_id}, keeping default (0.5): {e}")
        # 🔁 actualizar memoria
        link_sp_positions[link_id].append(0.5)

RelPos failed for 879.0, keeping default (0.5): (-2147352567, 'Exception occurred.', (0, 'Visum.Visum.2401', 'AttValue failed: Stop point 879 cannot be moved,  as the distance to the next stop point is (878) < 0.001000.', None, 0, -2147352567), None)
RelPos failed for 2898.0, keeping default (0.5): (-2147352567, 'Exception occurred.', (0, 'Visum.Visum.2401', 'AttValue failed: Stop point 2898 cannot be moved,  as the distance to the next stop point is (2897) < 0.001000.', None, 0, -2147352567), None)
Error inserting 2903.0: (-2147352567, 'Exception occurred.', (0, 'Visum.Visum.2401', 'Stop point 2903 cannot be inserted,  as the distance to the next stop point is (2898) < 0.001000.', None, 0, -2147352567), None)
Error inserting 2906.0: (-2147352567, 'Exception occurred.', (0, 'Visum.Visum.2401', 'Stop point 2906 cannot be inserted,  as the distance to the next stop point is (2898) < 0.001000.', None, 0, -2147352567), None)
Error inserting 579.0: (-2147352567, 'Exception occurred.', (0, '

In [69]:
stops_not_inserted

[224.0,
 879.0,
 974.0,
 1294.0,
 1359.0,
 1550.0,
 1754.0,
 2035.0,
 2077.0,
 2099.0,
 2361.0,
 2646.0,
 2834.0,
 2897.0,
 2898.0,
 2903.0,
 2906.0,
 2945.0,
 3083.0,
 3282.0,
 3285.0,
 3610.0,
 3694.0,
 3724.0,
 3745.0,
 3818.0,
 3821.0,
 4001.0,
 4049.0,
 4126.0,
 4158.0,
 4275.0,
 4319.0,
 4345.0,
 4408.0,
 4409.0,
 4645.0,
 4848.0,
 4908.0,
 5052.0,
 5065.0,
 5347.0,
 5368.0,
 5430.0,
 5527.0,
 5533.0,
 5841.0,
 5858.0,
 5881.0,
 6023.0,
 6443.0,
 6647.0,
 0.0,
 8066.0,
 8098.0,
 8454.0,
 8784.0,
 8801.0,
 8939.0,
 8966.0,
 8971.0,
 8975.0,
 9194.0,
 9433.0,
 9439.0,
 9440.0,
 9579.0,
 9685.0,
 9686.0,
 9690.0,
 9718.0,
 9729.0,
 9733.0,
 9756.0,
 9765.0,
 10488.0,
 10489.0,
 10584.0,
 10755.0,
 10756.0,
 10757.0,
 10829.0,
 10830.0,
 10854.0,
 10898.0,
 12175.0,
 12355.0,
 12375.0,
 12380.0,
 12411.0,
 12412.0]